# Demo 3: Lebesgue Spaces and Tensor-Product Spectral Basis in 3D

This notebook introduces the `Lebesgue3D` function space and the `TensorProductProvider`
spectral basis in **pygeoinf3D**. The key ideas:

- `Lebesgue3D(box)` is a basis-free $L^2$ Hilbert space: inner products and norms are
  computed by numerical integration, with no finite-dimensional component representation.
- `TensorProductProvider` provides an orthonormal spectral basis of eigenfunctions of the
  3D Laplacian on a box domain with either Dirichlet or Neumann boundary conditions.
- Manual projection onto the spectral basis allows function approximation and reconstruction.

**Sections:**
1. Imports and Setup
2. Basis-Free Lebesgue Space
3. TensorProductProvider: Spectral Basis
4. Visualising Eigenfunctions
5. Manual Projection and Reconstruction
6. Gram Matrix Heatmap
7. Neumann Boundary Conditions
8. Feature Summary

## 1. Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pygeoinf3d import BoxDomain, Lebesgue3D, TensorProductProvider, Function3D
from pygeoinf3d.plot import plot_function3d_axis_slice

# Unit cube [0,1]^3
box = BoxDomain(np.zeros(3), np.ones(3))

bbox = box.bounding_box()
print('Box domain:', box)
print('Low corner :', bbox.low)
print('High corner:', bbox.high)
print('Extents    :', box.extents)

## 2. Basis-Free Lebesgue Space

`Lebesgue3D(box)` constructs an $L^2(\Omega)$ Hilbert space without any spectral
representation. Its dimension is reported as **0** to indicate infinite-dimensionality
with no finite basis attached. Despite this, inner products and norms are fully
functional and use numerical quadrature under the hood.

**Key formulas:**
$$
\langle f, g \rangle_{L^2} = \int_\Omega f(\mathbf{x})\, g(\mathbf{x})\, d\mathbf{x},
\qquad
\|f\|_{L^2} = \sqrt{\langle f, f \rangle}
$$

For $f(\mathbf{x}) = \sin(\pi x_1)$ on $[0,1]^3$:
$$
\langle f, f \rangle = \int_0^1\!\int_0^1\!\int_0^1 \sin^2(\pi x_1)\, dx_1\, dx_2\, dx_3
= \int_0^1 \sin^2(\pi t)\, dt = \tfrac{1}{2}
$$

And $\langle \sin(\pi x_1), \cos(\pi x_1) \rangle = 0$ by symmetry over $[0,1]$.

In [ ]:
# 2a. Create the basis-free L2 space
space = Lebesgue3D(box)
print(f'space.dim = {space.dim}  (0 means basis-free / infinite-dimensional)')

# 2b. Define vectorised Function3D objects
# Callable signature: f(pts: (N,3)) -> (N,) array
def f_callable(pts):
    return np.sin(np.pi * pts[:, 0])

def g_callable(pts):
    return np.cos(np.pi * pts[:, 0])

f = Function3D(box, evaluate_callable=f_callable)
g = Function3D(box, evaluate_callable=g_callable)

# 2c. Inner products
ip_ff = space.inner_product(f, f)
ip_fg = space.inner_product(f, g)
print(f'\n<f, f> = {ip_ff:.8f}  (expected: 0.5)')
print(f'<f, g> = {ip_fg:.3e}  (expected: ~0, orthogonal)')

# 2d. Norm: verify norm^2 = inner_product(f,f)
nf = space.norm(f)
print(f'\nnorm(f)   = {nf:.8f}  (expected: {np.sqrt(0.5):.8f})')
print(f'norm(f)^2 = {nf**2:.8f}  == <f,f> = {ip_ff:.8f}')

## 3. TensorProductProvider: Spectral Basis

`TensorProductProvider` builds an orthonormal basis of $L^2([0,1]^3)$ by taking
tensor products of 1D Laplacian eigenfunctions. With `truncation=N`, it retains
the $N$ lowest-frequency modes in each coordinate direction, giving $N^3$ basis functions.

For **Dirichlet** BCs on $[0, L]$:
$$
\phi_n(t) = \sqrt{\frac{2}{L}}\,\sin\!\left(\frac{n\pi t}{L}\right), \quad n=1,2,\ldots
$$
with eigenvalues $\lambda_n = \left(\frac{n\pi}{L}\right)^2$.

The 3D eigenfunctions are products $\Phi_{i,j,k}(\mathbf{x}) = \phi_i(x_1)\phi_j(x_2)\phi_k(x_3)$,
ordered by eigenvalue $\lambda_{ijk} = \lambda_i + \lambda_j + \lambda_k$.

**Note (known limitation):** `space.to_components(f)` and `space.from_components(c)`
are not available — manual projection loops must be used instead.

In [ ]:
# 3a. Create the provider (truncation=3 -> 3^3=27 basis functions)
prov3 = TensorProductProvider(space, truncation=3, bc_type='dirichlet')
print(f'prov3.dim = {prov3.dim}  (expected: 27 = 3^3)')

# 3b. First 5 eigenvalues
print('\nFirst 5 eigenvalues (sorted ascending):')
for i in range(5):
    print(f'  lambda_{i} = {prov3.get_eigenvalue(i):.6f}')

# 3c. 5x5 Gram matrix G[i,j] = <phi_i, phi_j> -> should be ~delta_ij
print('\nComputing 5x5 Gram matrix...')
n_check = 5
G5 = np.zeros((n_check, n_check))
for i in range(n_check):
    phi_i = prov3.get_eigenfunction(i)
    for j in range(n_check):
        phi_j = prov3.get_eigenfunction(j)
        G5[i, j] = space.inner_product(phi_i, phi_j)

print('5x5 Gram matrix (should be approx I):')
print(np.round(G5, 6))

# 3d. Heatmap
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(G5, cmap='RdBu_r', vmin=-0.1, vmax=1.1)
ax.set_title('5x5 Gram matrix <phi_i, phi_j>')
ax.set_xlabel('j'); ax.set_ylabel('i')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Verify
off_diag_max = np.max(np.abs(G5 - np.eye(n_check)))
print(f'\nMax |G5 - I| = {off_diag_max:.2e}  (should be < 1e-10)')

## 4. Visualising Eigenfunctions

We plot $z=0.5$ slices of the first 8 eigenfunctions ($2^3 = 8$ with `truncation=2`).
Each panel shows the function value on the $xy$-plane at $z = 0.5$.

The pattern of increasing oscillations reflects the ascending eigenvalue ordering:
$\phi_0$ is the fundamental mode (one lobe in each direction), while higher modes
introduce more oscillations in individual coordinate directions.

**Note:** Some slices appear near-zero (colorbar scale ~1e-16). This is physically
correct: eigenfunctions with a 2nd-order mode in $z$ contain $\sin(2\pi z)$, which
is exactly zero at $z = 0.5$.

In [ ]:
# 4. Plot first 8 eigenfunctions (truncation=2, dim=8) as z=0.5 slices
prov2 = TensorProductProvider(space, truncation=2, bc_type='dirichlet')
print(f'prov2.dim = {prov2.dim}  (expected: 8 = 2^3)')

n_funcs = prov2.dim  # 8
fig, axes = plt.subplots(3, 3, figsize=(10, 9))

for idx in range(9):
    ax = axes[idx // 3, idx % 3]
    if idx < n_funcs:
        phi = prov2.get_eigenfunction(idx)
        lam = prov2.get_eigenvalue(idx)
        plot_function3d_axis_slice(phi, axis='z', coord=0.5, ax=ax, show_plot=False)
        ax.set_title(f'phi_{idx}  (lambda={lam:.2f})', fontsize=9)
    else:
        ax.axis('off')  # last slot empty for 8 functions in 3x3 grid

plt.suptitle('Eigenfunctions phi_0..phi_7 at z=0.5 (truncation=2, Dirichlet)', y=1.01)
plt.tight_layout()
plt.show()

## 5. Manual Projection and Reconstruction

Given any $f \in L^2(\Omega)$, we can project it onto the spectral basis:
$$
c_i = \langle \phi_i, f \rangle, \qquad
f_{\rm approx} = \sum_{i=0}^{N-1} c_i\, \phi_i
$$

This is the **only valid approach** with the current `TensorProductProvider` API —
`space.to_components(f)` is not available.

We use $f(\mathbf{x}) = \sin(\pi x)\sin(\pi y)\sin(\pi z)$, which is proportional to the
first Dirichlet eigenfunction $\phi_0 = \sqrt{8}\sin(\pi x)\sin(\pi y)\sin(\pi z)$.
Therefore $c_0 = \langle\phi_0, f\rangle = \sqrt{8}\cdot(1/2)^3 = 1/\sqrt{8} \approx 0.354$,
all other coefficients are zero, and the reconstruction is exact to machine precision.

In [ ]:
# 5a. Test function: sin(pi x)sin(pi y)sin(pi z) - vectorised
def f_test_callable(pts):
    return (np.sin(np.pi * pts[:, 0])
            * np.sin(np.pi * pts[:, 1])
            * np.sin(np.pi * pts[:, 2]))

f_test = Function3D(box, evaluate_callable=f_test_callable)

# 5b. TensorProductProvider with truncation=4 (64 basis functions)
# NOTE: SLOW CELL - computing 64 inner products via numerical integration
prov4 = TensorProductProvider(space, truncation=4, bc_type='dirichlet')
print(f'prov4.dim = {prov4.dim}  (expected: 64 = 4^3)')
print('Computing projection coefficients (64 inner products)...')

coeffs = np.array([
    space.inner_product(prov4.get_eigenfunction(i), f_test)
    for i in range(prov4.dim)
])
# f_test = sin(pi x)sin(pi y)sin(pi z) is exactly phi_0 / sqrt(8) (unnormalized)
# phi_0 = sqrt(8) * sin(pi x)sin(pi y)sin(pi z), so c_0 = <phi_0, f> = 1/sqrt(8)
expected_c0 = 1.0 / np.sqrt(8)
print(f'Done. c_0 = {coeffs[0]:.6f}  (expected approx {expected_c0:.6f} = 1/sqrt(8))')
print(f'Non-zero coefficients (|c_i|>1e-8): {np.sum(np.abs(coeffs) > 1e-8)}')

# 5c. Reconstruct: f_approx = sum c_i phi_i
f_approx = coeffs[0] * prov4.get_eigenfunction(0)
for i in range(1, prov4.dim):
    if abs(coeffs[i]) > 1e-14:
        f_approx = f_approx + coeffs[i] * prov4.get_eigenfunction(i)

# 5d. Relative reconstruction error
err_func = f_test - f_approx
rel_err = space.norm(err_func) / space.norm(f_test)
print(f'\nRelative reconstruction error ||f - f_approx||/||f|| = {rel_err:.3e}')
print('(Expected: close to machine precision - f_test is exactly a basis function)')

In [ ]:
# 5e. Visualise original vs reconstruction at z=0.5
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

plot_function3d_axis_slice(f_test, axis='z', coord=0.5, ax=axes[0], show_plot=False)
axes[0].set_title('Original f = sin(pi x)sin(pi y)sin(pi z) at z=0.5')

plot_function3d_axis_slice(f_approx, axis='z', coord=0.5, ax=axes[1], show_plot=False)
axes[1].set_title(f'Reconstruction (truncation=4) at z=0.5\nRel. error = {rel_err:.2e}')

plt.tight_layout()
plt.show()

## 6. Gram Matrix Heatmap

The full $27 \times 27$ Gram matrix $G_{ij} = \langle \phi_i, \phi_j \rangle$ for
the `truncation=3` Dirichlet basis should be the identity matrix.

Any deviation from $\delta_{ij}$ quantifies integration error in the quadrature.

**Note:** This cell computes $27(27+1)/2 = 378$ unique inner products (exploiting symmetry).

In [ ]:
# 6. Full 27x27 Gram matrix for truncation=3 Dirichlet basis
# NOTE: SLOW CELL - 27*(27+1)/2 = 378 unique inner product evaluations
n = prov3.dim  # 27
print(f'Computing {n}x{n} = {n*n} entries ({n*(n+1)//2} unique by symmetry)...')

# Pre-fetch all eigenfunctions to avoid recomputing
phis = [prov3.get_eigenfunction(i) for i in range(n)]

G = np.zeros((n, n))
for i in range(n):
    for j in range(i, n):   # exploit symmetry
        val = space.inner_product(phis[i], phis[j])
        G[i, j] = val
        G[j, i] = val

print('Done.')

# Display as heatmap
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(G, cmap='RdBu_r', vmin=-0.05, vmax=1.05)
ax.set_title('27x27 Gram matrix G[i,j]=<phi_i, phi_j> (Dirichlet, truncation=3)')
ax.set_xlabel('j'); ax.set_ylabel('i')
plt.colorbar(im, ax=ax, label='<phi_i, phi_j>')
plt.tight_layout()
plt.show()

# Verify closeness to identity
off_diag_err = np.max(np.abs(G - np.eye(n)))
diag_err = np.max(np.abs(np.diag(G) - 1.0))
print(f'\nMax |G - I| (all entries) = {off_diag_err:.2e}')
print(f'Max |diag(G) - 1|          = {diag_err:.2e}')

## 7. Neumann Boundary Conditions

`TensorProductProvider` also supports **Neumann** boundary conditions, where the
normal derivative vanishes at the boundary. The 1D eigenfunctions are:
$$
\phi_0(t) = \sqrt{\frac{1}{L}}, \quad
\phi_n(t) = \sqrt{\frac{2}{L}}\cos\!\left(\frac{n\pi t}{L}\right), \quad n=1,2,\ldots
$$
with eigenvalues $\lambda_0 = 0$, $\lambda_n = \left(\frac{n\pi}{L}\right)^2$.

Key differences from Dirichlet:
- The lowest eigenvalue is **0** (constant mode), not $3\pi^2$
- The fundamental mode $\phi_0$ is a constant function (no oscillations)
- Eigenfunctions are cosines, not sines

In [ ]:
# 7a. Create Neumann provider (truncation=3)
prov_neu = TensorProductProvider(space, truncation=3, bc_type='neumann')
print(f'Neumann provider dim = {prov_neu.dim}  (expected: 27)')

# 7b. Compare first 5 eigenvalues side-by-side
print('\nEigenvalue comparison (first 5):')
print(f'{"i":>3}  {"Dirichlet":>14}  {"Neumann":>14}')
print('-' * 36)
for i in range(5):
    lam_d = prov3.get_eigenvalue(i)
    lam_n = prov_neu.get_eigenvalue(i)
    print(f'{i:>3}  {lam_d:>14.6f}  {lam_n:>14.6f}')

print('\n(Neumann lambda_0 should be ~0.0 -- constant mode)')
print(f'(Dirichlet lambda_0 should be ~{3*np.pi**2:.4f} -- 3*pi^2 for unit cube)')

# 7c. Visualise phi_0 for both side-by-side at z=0.5
phi0_dir = prov3.get_eigenfunction(0)
phi0_neu = prov_neu.get_eigenfunction(0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_function3d_axis_slice(phi0_dir, axis='z', coord=0.5, ax=axes[0], show_plot=False)
axes[0].set_title(f'Dirichlet phi_0 at z=0.5  (lambda={prov3.get_eigenvalue(0):.3f})')

plot_function3d_axis_slice(phi0_neu, axis='z', coord=0.5, ax=axes[1], show_plot=False)
axes[1].set_title(f'Neumann phi_0 at z=0.5  (lambda={prov_neu.get_eigenvalue(0):.3f})')

plt.suptitle('Dirichlet vs Neumann: fundamental eigenfunction phi_0', y=1.02)
plt.tight_layout()
plt.show()

## 8. Feature Summary

| Feature | API | Result |
|---------|-----|--------|
| Basis-free L² space | `Lebesgue3D(box)` | `space.dim == 0` |
| Numerical inner product | `space.inner_product(f, g)` | Uses `box.integrate_volume` |
| Numerical norm | `space.norm(f)` | `sqrt(<f,f>)` |
| Spectral basis (Dirichlet) | `TensorProductProvider(space, truncation=N, bc_type='dirichlet')` | $N^3$ sine eigenfunctions |
| Spectral basis (Neumann) | `TensorProductProvider(space, truncation=N, bc_type='neumann')` | $N^3$ cosine eigenfunctions |
| Get eigenfunction | `prov.get_eigenfunction(i)` | `Function3D` |
| Get eigenvalue | `prov.get_eigenvalue(i)` | `float` |
| Number of basis functions | `prov.dim` | $N^3$ |
| Manual projection | `c[i] = space.inner_product(prov.get_eigenfunction(i), f)` | Requires loop |
| Reconstruction | `sum(c[i] * prov.get_eigenfunction(i) for i in ...)` | `Function3D` arithmetic |
| Gram matrix | $G_{ij} = \langle\phi_i,\phi_j\rangle$ | $\approx I$ for orthonormal basis |
| Slice visualisation | `plot_function3d_axis_slice(phi, axis='z', coord=0.5)` | Matplotlib axes |

### Next Demo
**[3.1 — Kernel Functionals in 3D](3.1_kernel_functionals_3d_demo.ipynb)**